## 라쿠텐 top10 추출및 모든 리뷰데이터 추출코드

In [11]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import time
import random
import json
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
TARGET_URL       = "https://ranking.rakuten.co.jp/weekly/100944/p=1/"
RANK_SAVE_FILE   = "rakuten_rankings_current.jsonl"
REVIEW_SAVE_FILE = "rakuten_reviews_master.jsonl"

RAW_COOKIE = r"_ra=1773145336745|9810936d-5cda-4630-8e49-c98379684815; Rp=dc4c4487b0d3c4dce21f7d6bdbf70329f03486c8; rcxGlobal=d45a9a4c-4afa-4759-a9ed-229577264fb8; bm_mi=02819FD1874A677A9FC3219D127ECD18~YAAQB9ojF3ykzaGcAQAAMAK01x85bJZA06vzzbG4DIKAXg7mopEgiKN5lzrV/0AErGfSM50oZ5QfIMqbrcROt7DYSkrYw58Cz7vwEO2XkjKMkHGsBRiolwhSHY6BjhqOg8UrHpjdOwbKwGHWbgXrhYVrXbpqnuYxOnS6Rd4tbRsuPIrmZrskES38smu59dSiBK0bGYfeffgKl/2sz7WT7sEbAOAom3GeHhE5vSnLp8Unxda00YTc54V1R0SccenoAvO/8tAjlqf4fcZMyVfgWjmkkXpBvobpj5hyrWagxWfFFgn06n/L/mwrI22pddxKke0Bln2zVz7w9QvyrImNfBfU7zu1YGnnS6HKKDs9fU21qukz3/qjiTlpM3yQIgMrYmSo2aL/dooM6uWJl/I7WeDfGzeHGvHpBy6zNQ==~1; bm_sv=F3073B8724961BB7DCBE5A9516653181~YAAQbIj+eWOAwsucAQAA7Di01x/kkLOsyzmShVkJPcJf1O0TkzKdl7ZL7qgaen5E33IuCROVCPlrVdQr767lxL25yZRnHKhQTLV74swQf/WGDo3WVE2ehk0W7uv6G6PUyNE2N5dQyBl/0tnQF1TIL1GoIY1QVyysRmPhNLu9lndOUVbr8CgHlwjK+8cZfXcWG/rrLNFiEsHCtxhiQUijgpAP4TGE1raLDQ8pAaWOcclxAHnAjyBcQ2TkH+CwcSLNeCnH~1; ak_bmsc=6DA2C442401E92959215D4FC32333198~000000000000000000000000000000~YAAQB9ojF5AVzqGcAQAA0cu01x9fj3L5e+d9RAJ1hoLlH3K8NWtbYVM99UU1cPhBjs0l+RT4L93zSS2KCBZ2cF79c9IkAVFdcgGHbxP50Uf8nlO5I6u+fsByVK6zS4EhUNSCSizuhMPBdf0lHio/+lSN+8w0fTRgZfWwPNbWNRlfyg8SAriAihvo7Qf2txj9kJZGnaHPqgkCZXs11UwhehZ5Ak3MzYtTcojSc91bzqp+zzmmEbyhuYzqW5FWGr6d5k6LF157rv/BKCBjyW2dtk7TbRgJKgQGZz/gPYD8n8mDMroN36ptPxuByW5GOOwTwoCs8tDnUOjgOtm8M7mmdRNKlpJsUSJoOylT7fkQoeVPZkHsLG1j5QOPG60+P7zXsIeB4pddGO/PKsDF8HPAgLpOVb2Dgqz56kyCR4vfPNXZLawqlDXl8+8HXh2AL5KJmQIZMy0XvT/jsyXGkSBk6KO8l/jsyQp1AG570i/fWulONUZGzkQKF86JqOOaoDS7dUrDgp1mqvDR4lwtHSs=; krt_rewrite_uid=f8b15f33-cc3e-45c0-b1be-9a29a4cff1d2; Re=31.1.5.0.0.216348.3-31.1.5.0.0.216348.3; rat_v=05252dcb01d4e6863512b1a3e769b00e711919a"
SAFE_COOKIE = RAW_COOKIE.encode('utf-8').decode('latin-1', 'ignore')

REVIEW_API_URL = "https://web-gateway.rakuten.co.jp/review/itemshopreviewlist/get/v1"
REVIEW_HEADERS = {
    "authkey"     : "isrlPcMjUuXCVBUTVh91ZcHEfoI45CmPR",
    "content-type": "application/json; charset=UTF-8",
    "accept"      : "application/json, text/plain, */*",
    "user-agent"  : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    "cookie"      : SAFE_COOKIE,
    "referer"     : "https://review.rakuten.co.jp/",
}

# ── 별점별 수집 전략 ──────────────────────────────────────────────
# 1~3점: 전체 수집 (100페이지 제한 내 수집 가능)
# 4점  : 전체 수집
# 5점  : 전체 수집
# ※ 100페이지(30개×100=3000개)를 초과하는 경우 자동 종료
MAX_PAGES = 100


# ── 함수: 별점 필터별 리뷰 수집 ──────────────────────────────────
def get_reviews_by_rating(shop_id, item_id, rating_filter):
    """
    rating_filter: "1", "2", "3", "4", "5" 또는 "" (전체)
    """
    all_reviews = []
    page        = 1
    has_next    = True

    while has_next and page <= MAX_PAGES:
        payload = {
            "common": {
                "params" : {"device": "pc"},
                "include": ["itemReviewList"],
            },
            "features": {
                "itemReviewList": {
                    "params": {
                        "shopId"             : int(shop_id),
                        "itemId"             : int(item_id),
                        "sort"               : "",
                        "page"               : str(page),
                        "hits"               : 30,
                        "filter"             : {
                            "rating"   : rating_filter,   # ← 별점 필터 핵심
                            "mediaOnly": "false",
                            "ageRange" : "",
                            "sex"      : "",
                        },
                        "includePickupReview": True,
                    }
                }
            },
        }
        try:
            resp = requests.post(REVIEW_API_URL, headers=REVIEW_HEADERS, json=payload, timeout=15)
            if resp.status_code not in [200, 207]:
                print(f"\n    ❌ {page}p 중단 (코드: {resp.status_code})")
                break

            data     = resp.json()
            res_body = data.get("body", {}).get("itemReviewList", {})
            res_data = res_body.get("data", {})
            reviews  = res_data.get("reviews", [])

            if not reviews:
                break

            for rev in reviews:
                all_reviews.append({
                    "shop_id"      : shop_id,
                    "item_id"      : item_id,
                    "rating_filter": rating_filter or "all",
                    "Page"         : page,
                    "Nickname"     : rev.get("nickname"),
                    "Rating"       : rev.get("rating"),
                    "Body"         : rev.get("body"),
                    "PostDate"     : rev.get("postDate"),
                    "Age"          : f"{rev.get('ageRange', '')}{rev.get('ageSuffix', '')}",
                    "Sex"          : rev.get("sex"),
                    "Sku"          : rev.get("skuInfo"),
                })

            has_next = res_data.get("hasNextPage", False)
            page += 1
            time.sleep(1.8)

        except Exception as e:
            print(f"\n    ❌ 에러: {e}")
            break

    return all_reviews


# ── 함수: 별점 전략에 따라 전체 리뷰 수집 ────────────────────────
def get_rakuten_all_reviews(shop_id, item_id, product_name):
    all_reviews = []

    print(f"\n  🚀 [{product_name[:35]}] 리뷰 수집 시작...")

    # ── 전략: 1~3점은 각각 개별 수집, 5점은 전체 수집 ─────────────
    # 1점, 2점, 3점 → 각각 100페이지 이내이므로 별점별 수집
    for star in ["1", "2", "3"]:
        print(f"    ⭐{star}점 수집 중...", end="\r")
        reviews = get_reviews_by_rating(shop_id, item_id, star)
        all_reviews.extend(reviews)
        print(f"    ⭐{star}점: {len(reviews)}개 수집 완료")
        time.sleep(1.0)

    # 4점 → 별점별 수집
    print(f"    ⭐4점 수집 중...", end="\r")
    reviews = get_reviews_by_rating(shop_id, item_id, "4")
    all_reviews.extend(reviews)
    print(f"    ⭐4점: {len(reviews)}개 수집 완료")
    time.sleep(1.0)

    # 5점 → 전체 수집 (가장 많으므로 100페이지 한도 내에서 최대한)
    print(f"    ⭐5점 수집 중...", end="\r")
    reviews = get_reviews_by_rating(shop_id, item_id, "5")
    all_reviews.extend(reviews)
    print(f"    ⭐5점: {len(reviews)}개 수집 완료 {'(100p 한도 도달)' if len(reviews) >= 3000 else ''}")

    print(f"\n  ✨ 전체 수집 완료: {len(all_reviews)}개 (⭐1~5점 합산)")
    return all_reviews


# ── 메인 ─────────────────────────────────────────────────────────
def fetch_rakuten_data():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    driver = uc.Chrome(options=options)

    try:
        print("📡 라쿠텐 랭킹 페이지 접속 중...")
        driver.get(TARGET_URL)
        time.sleep(random.uniform(7, 10))

        if "アクセスが集中" in driver.page_source:
            print("⚠️ 차단 감지. rakuten_debug.html 저장됨")
            with open("rakuten_debug.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            return

        soup      = BeautifulSoup(driver.page_source, "html.parser")
        all_items = soup.select("div.rnkRanking_top3box, div.rnkRanking_after4box")
        print(f"📦 상품 카드 {len(all_items)}개 발견")

        if not all_items:
            with open("rakuten_debug.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print("⚠️ 상품 카드 없음. rakuten_debug.html 저장됨")
            return

        rank_data_list    = []
        all_review_master = []
        rank_count        = 1

        for item in all_items:
            if rank_count > 10:
                break
            try:
                rank_icon = item.select_one(".rnkRanking_rankIcon img")
                disp_rank = item.select_one(".rnkRanking_dispRank")
                if rank_icon:
                    alt      = rank_icon.get("alt", "")
                    rank_num = int(alt.replace("位", "")) if "位" in alt else rank_count
                elif disp_rank:
                    t        = disp_rank.get_text(strip=True).replace("位", "")
                    rank_num = int(t) if t.isdigit() else rank_count
                else:
                    rank_num = rank_count

                title_tag = item.select_one(".rnkRanking_itemName a")
                if not title_tag:
                    continue
                title       = title_tag.get_text(strip=True)
                product_url = title_tag.get("href", "")

                shop_tag  = item.select_one(".rnkRanking_shop a")
                shop_name = shop_tag.get_text(strip=True) if shop_tag else "N/A"

                price_tag = item.select_one(".rnkRanking_price")
                price     = price_tag.get_text(strip=True) if price_tag else "N/A"

                stars_on   = len(item.select(".rnkRanking_starON"))
                stars_half = len(item.select(".rnkRanking_starHALF"))
                rating     = stars_on + (0.5 if stars_half else 0)

                review_link = item.select_one("a[href*='review.rakuten.co.jp/item']")
                reviews_cnt = 0
                shop_id = item_id = ""
                if review_link:
                    rv_text     = review_link.get_text(strip=True)
                    reviews_cnt = int(
                        rv_text.replace("レビュー(", "").replace("件)", "").replace(",", "").strip()
                    ) if "レビュー" in rv_text else 0
                    href    = review_link.get("href", "")
                    parts   = href.rstrip("/").split("/")
                    id_part = next((p for p in reversed(parts) if "_" in p), "")
                    if id_part:
                        shop_id, item_id = id_part.split("_", 1)

                trend_img = item.select_one(".rnkRanking_preRank img")
                trend = "N/A"
                if trend_img:
                    alt   = trend_img.get("alt", "")
                    trend = "Up" if "Up" in alt else ("Down" if "Down" in alt else "Stay")

                rank_data_list.append({
                    "rank"        : rank_num,
                    "title"       : title,
                    "shop_name"   : shop_name,
                    "rating"      : rating,
                    "reviews"     : reviews_cnt,
                    "price"       : price,
                    "trend"       : trend,
                    "url"         : product_url,
                    "shop_id"     : shop_id,
                    "item_id"     : item_id,
                    "platform"    : "Rakuten",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })
                print(f"\n📍 {rank_num:>2}위 [{trend}] {shop_name[:15]} | {title[:30]} | ⭐{rating} ({reviews_cnt}건)")

                if shop_id and item_id:
                    reviews = get_rakuten_all_reviews(shop_id, item_id, title)
                    all_review_master.extend(reviews)
                    print(f"  → 누적 리뷰: {len(all_review_master)}개")
                else:
                    print(f"  ⚠️ shop_id/item_id 없음 — 리뷰 수집 스킵")

                rank_count += 1

            except Exception as e:
                print(f"⚠️ {rank_count}위 파싱 오류: {e}")
                rank_count += 1
                continue

        with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
            for entry in rank_data_list:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
            for review in all_review_master:
                f.write(json.dumps(review, ensure_ascii=False) + "\n")

        print()
        print("=" * 70)
        print("🏆 라쿠텐 스킨케어 랭킹 Top 10 (주간)")
        print("=" * 70)
        for item in rank_data_list:
            print(f"{item['rank']:>2}위 [{item['trend']}] | {item['shop_name'][:15]:<15} | {item['title'][:25]:<25} | ⭐{item['rating']} | {item['price']}")
        print("=" * 70)
        print(f"\n📊 JSONL 저장 완료")
        print(f"  - 순위 파일 : {RANK_SAVE_FILE} ({len(rank_data_list)}개 상품)")
        print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(all_review_master)}개 리뷰)")

    finally:
        driver.quit()


if __name__ == "__main__":
    fetch_rakuten_data()

📡 라쿠텐 랭킹 페이지 접속 중...
📦 상품 카드 80개 발견

📍  1위 [Up] VTcosmetic楽天市場店 | ＼最大49％OFF＋ギフト＋送料無料／総合ランキング1位【V | ⭐4.5 (7886건)

  🚀 [＼最大49％OFF＋ギフト＋送料無料／総合ランキング1位【VT 大容量] 리뷰 수집 시작...
    ⭐1점: 135개 수집 완료
    ⭐2점: 84개 수집 완료
    ⭐3점: 247개 수집 완료
    ⭐4점: 1194개 수집 완료
    ⭐5점: 3000개 수집 완료 (100p 한도 도달)

  ✨ 전체 수집 완료: 4660개 (⭐1~5점 합산)
  → 누적 리뷰: 4660개

📍  2위 [Down] アテニア公式ショップ　楽天市場 | 【ポイント10倍！〜3月11日1:59】スキンクリア クレン | ⭐4.5 (44082건)

  🚀 [【ポイント10倍！〜3月11日1:59】スキンクリア クレンズ オイル] 리뷰 수집 시작...
    ⭐1점: 50개 수집 완료
    ⭐2점: 70개 수집 완료
    ⭐3점: 567개 수집 완료
    ⭐4점: 3000개 수집 완료
    ⭐5점: 3000개 수집 완료 (100p 한도 도달)

  ✨ 전체 수집 완료: 6687개 (⭐1~5점 합산)
  → 누적 리뷰: 11347개

📍  3위 [Up] VTcosmetic楽天市場店 | 【クーポン付き】＼最大68％OFF＋現品ギフト＋送料無料／【 | ⭐4.5 (1520건)

  🚀 [【クーポン付き】＼最大68％OFF＋現品ギフト＋送料無料／【VT公式】] 리뷰 수집 시작...
    ⭐1점: 24개 수집 완료
    ⭐2점: 15개 수집 완료
    ⭐3점: 49개 수집 완료
    ⭐4점: 266개 수집 완료
    ⭐5점: 1219개 수집 완료 

  ✨ 전체 수집 완료: 1573개 (⭐1~5점 합산)
  → 누적 리뷰: 12920개

📍  4위 [Up] 【公式】Yunth Store | 【P30%還元+セット9日23:59マデ】【公式】Yunth | ⭐4.5 (43354건)

  🚀 [【P30%還元+セッ

## 팀원에게 받은 번역코드


In [1]:
from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json, time, re
from tqdm import tqdm

# ── 설정 ─────────────────────────────────────────────────────
INPUT_FILE  = './rakuten_reviews_master.jsonl'
OUTPUT_FILE = 'rakuten_master_translated_jp_ko.jsonl'
BODY_COL    = 'Body'
N_SAMPLE    = None

CHUNK_SIZE  = 5
MAX_WORKERS = 4
MAX_RETRIES = 3
RETRY_SLEEP = 2.0
CHUNK_DELAY = 0.5


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 줄번호 방식 파싱 / 조립
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_numbered(text: str, expected_n: int) -> list[str]:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    result  = {int(idx): body.strip() for idx, body in pattern.findall(text)}
    return [result.get(i + 1, "") for i in range(expected_n)]

def build_numbered(texts: list[str]) -> str:
    return "\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 단건 번역 (fallback 전용)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not text.strip():
        return ""
    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source=src, target=tgt).translate(text)
            if result and result.strip():
                return result.strip()
        except Exception:
            pass
        time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 청크 번역 — 1단계 (일본어 → 영어)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_chunk_ja_en(texts: list[str]) -> list[str]:
    if not texts:
        return []
    joined = build_numbered(texts)
    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source='ja', target='en').translate(joined)
            if not result:
                raise ValueError("빈 응답")
            parts = parse_numbered(result, len(texts))
            if all(p for p in parts):
                return parts
            for i, p in enumerate(parts):
                if not p:
                    parts[i] = translate_single(texts[i], 'ja', 'en')
            return parts
        except Exception:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return [translate_single(t, 'ja', 'en') for t in texts]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 청크 번역 — 2단계 (영어 → 한국어)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_chunk_en_ko(texts: list[str]) -> list[str]:
    if not texts:
        return []
    joined = build_numbered(texts)
    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source='en', target='ko').translate(joined)
            if not result:
                raise ValueError("빈 응답")
            parts = parse_numbered(result, len(texts))
            if all(p for p in parts):
                return parts
            for i, p in enumerate(parts):
                if not p:
                    parts[i] = translate_single(texts[i], 'en', 'ko')
            return parts
        except Exception:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return [translate_single(t, 'en', 'ko') for t in texts]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2단계 통합
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_chunk_2step(texts: list[str]) -> tuple[list[str], list[str]]:
    en_texts = translate_chunk_ja_en(texts)
    time.sleep(CHUNK_DELAY)
    ko_texts = translate_chunk_en_ko(en_texts)
    return en_texts, ko_texts


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인 파이프라인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print("📥 데이터 로드...")
    raw_records = []
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:   # [BUG-1] 들여쓰기 복구
        for line in f:
            line = line.strip()
            if line:
                raw_records.append(json.loads(line))

    df = pd.DataFrame(raw_records)                        # [BUG-2] main() 안으로 복구

    if N_SAMPLE:
        df = df.head(N_SAMPLE)

    print(f"   총 {len(df):,}건  /  컬럼: {list(df.columns)}")

    # ── 청크 분할 및 번역 ────────────────────────────────────
    texts    = df[BODY_COL].fillna("").tolist()
    chunks   = [texts[i:i+CHUNK_SIZE] for i in range(0, len(texts), CHUNK_SIZE)]
    n_chunks = len(chunks)

    print(f"\n🚀 2단계 번역 시작 (일→영→한)")
    print(f"   {len(texts):,}건  /  {n_chunks}개 청크  /  workers={MAX_WORKERS}")
    print(f"   CHUNK_SIZE={CHUNK_SIZE}  CHUNK_DELAY={CHUNK_DELAY}s")
    print(f"   예상 API 호출: 최대 {n_chunks * 2}회")

    start_time = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chunk_results = list(tqdm(
            executor.map(translate_chunk_2step, chunks),
            total=n_chunks,
            desc="번역 (ja→en→ko)",
            unit="chunk"
        ))

    elapsed = time.time() - start_time
    print(f"\n⏱  번역 완료: {elapsed:.1f}초 ({elapsed/60:.1f}분)")

    # ── 결과 병합 ────────────────────────────────────────────
    en_results: list[str] = []
    ko_results: list[str] = []
    for en_chunk, ko_chunk in chunk_results:
        en_results.extend(en_chunk)
        ko_results.extend(ko_chunk)

    df['Body_en'] = en_results[:len(df)]
    df['Body_ko'] = ko_results[:len(df)]

    # ── 성공률 ───────────────────────────────────────────────
    def success_mask(col):
        return df[col].str.strip().ne("") & df[col].ne("번역실패")

    en_ok = success_mask('Body_en')
    ko_ok = success_mask('Body_ko')

    print(f"\n📊 번역 결과")
    print(f"   1단계 (ja→en) 성공률: {en_ok.mean()*100:.1f}%  ({en_ok.sum():,}/{len(df):,}건)")
    print(f"   2단계 (en→ko) 성공률: {ko_ok.mean()*100:.1f}%  ({ko_ok.sum():,}/{len(df):,}건)")
    print(f"   처리 속도: {len(df)/elapsed:.1f}건/초")

    # ── 저장 ─────────────────────────────────────────────────
    print(f"\n💾 저장 중: {OUTPUT_FILE}")
    output_records = json.loads(df.to_json(orient='records', force_ascii=False))
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in output_records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"✅ 저장 완료: {OUTPUT_FILE}")

    # ── 샘플 3건 ─────────────────────────────────────────────
    print(f"\n📋 샘플 3건:")
    for _, row in df.head(3).iterrows():
        print(f"   item_id : {row.get('item_id', 'N/A')}")
        print(f"   rating  : ★{int(row.get('Rating', 0))}")
        print(f"   원문(ja): {str(row[BODY_COL])[:60]}...")
        print(f"   영어(en): {str(row['Body_en'])[:60]}...")
        print(f"   한국(ko): {str(row['Body_ko'])[:60]}...")
        print("   " + "─" * 52)

if __name__ == "__main__":
    main()

📥 데이터 로드...
   총 36,613건  /  컬럼: ['shop_id', 'item_id', 'rating_filter', 'Page', 'Nickname', 'Rating', 'Body', 'PostDate', 'Age', 'Sex', 'Sku']

🚀 2단계 번역 시작 (일→영→한)
   36,613건  /  7323개 청크  /  workers=4
   CHUNK_SIZE=5  CHUNK_DELAY=0.5s
   예상 API 호출: 최대 14646회


번역 (ja→en→ko):   0%|          | 0/7323 [00:03<?, ?chunk/s]


KeyboardInterrupt: 